# OCR and Tesseract

# Text recognition in images

## Installing Tesseract

- Documentation: https://pypi.org/project/pytesseract/

In [ ]:
!sudo apt install tesseract-ocr
!pip install pytesseract

## Importing the libraries

In [ ]:
import pytesseract
import numpy as np
import cv2 # OpenCV
from google.colab.patches import cv2_imshow

## Reading the image

In [ ]:
img = cv2.imread('/content/ocr01.jpg')
cv2_imshow(img) # BGR -> RGB

In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
text = pytesseract.image_to_string(rgb)

In [ ]:
print(text)

## Support for other languages

In [ ]:
img = cv2.imread('/content/language02.jpg')
cv2_imshow(img)

In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
text = pytesseract.image_to_string(rgb)
print(text)

In [ ]:
!tesseract --list-langs

In [ ]:
!apt-get install tesseract-ocr-por # Portuguese

In [ ]:
!tesseract --list-langs

In [ ]:
text = pytesseract.image_to_string(rgb, lang='por')
print(text)

In [ ]:
!mkdir tessdata

In [ ]:
!wget -O ./tessdata/por.traineddata https://github.com/tesseract-ocr/tessdata/blob/main/por.traineddata?raw=true

In [ ]:
config_tesseract = '--tessdata-dir tessdata'
text = pytesseract.image_to_string(rgb, lang='por', config=config_tesseract)
print(text)

In [ ]:
!wget -O ./tessdata/eng.traineddata https://github.com/tesseract-ocr/tessdata/blob/main/eng.traineddata?raw=true

In [ ]:
!ls tessdata/

## Parameters

### Page segmentation modes (PSM)

In [ ]:
!tesseract --help-psm

In [ ]:
img = cv2.imread('/content/pageFromBook.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
config_tesseract = '--tessdata-dir tessdata --psm 6'
text = pytesseract.image_to_string(rgb, lang='por', config=config_tesseract)
print(text)

In [ ]:
config_tesseract = '--tessdata-dir tessdata --psm 6'
text = pytesseract.image_to_string(rgb, lang='por', config=config_tesseract)
print(text)

In [ ]:
img = cv2.imread('/content/exit.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
config_tesseract = '--tessdata-dir tessdata --psm 7'
text = pytesseract.image_to_string(rgb, lang='por', config=config_tesseract)
print(text)

### Page orientation

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
img = Image.open('/content/anotherBook.jpg')
plt.imshow(img);

In [ ]:
print(pytesseract.image_to_osd(img))

# Selection of texts



In [ ]:
from pytesseract import Output

In [ ]:
img = cv2.imread('/content/ocr01.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
config_tesseract = '--tessdata-dir tessdata'
result = pytesseract.image_to_data(rgb, config=config_tesseract, lang='eng', output_type=Output.DICT)
result

- block_num = Current block number. When Tesseract performs the detections, it divides the image into several regions, which can vary according to the PSM parameters and also other criteria of the algorithm. Each block is a region

- conf = prediction confidence (from 0 to 100. -1 means no text was recognized)

- height = height of detected block of text (bounding box)

- left = x coordinate where the bounding box starts

- level = the level corresponds to the category of the detected block. There are 5 possible values:
  1. page
  2. block
  3. paragraph
  4. line
  5. word

Therefore, if 5 is returned, it means that the detected block is text, if it was 4, it means that a line was detected

- line_num = line number (starts from 0)

- page_num = the index of the page where the item was detected

- text = the recognition result

- top = y-coordinate where the bounding box starts

- width = width of the current detected text block

- word_num = word number (index) within the current block

In [ ]:
result['text'], len(result['text'])

In [ ]:
def bouding_box(result, img, i, color = (255,100,0)):
  x = result['left'][i]
  y = result['top'][i]
  w = result['width'][i]
  h = result['height'][i]

  cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)

  return x, y, img

In [ ]:
min_confidence = 40
img_copy = rgb.copy()
for i in range(0, len(result['text'])):
  #print(i)
  confidence = int(result['conf'][i])
  #print(confidence)
  if confidence > min_confidence:
    #print(confidence)
    x, y, img = bouding_box(result, img_copy, i)
    #print(x,y)
    text = result['text'][i]
    cv2.putText(img_copy, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,0,255))
cv2_imshow(img_copy)

In [ ]:
img = cv2.imread('/content/language02.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
config_tesseract = '--tessdata-dir tessdata'
result = pytesseract.image_to_data(rgb, config=config_tesseract, lang = 'por', output_type = Output.DICT)
result

In [ ]:
from PIL import ImageFont, ImageDraw, Image
font = '/content/calibri.ttf'

In [ ]:
def write_text(text, x, y, img, font, font_size = 32):
  font = ImageFont.truetype(font, font_size)
  img_pil = Image.fromarray(img)
  draw = ImageDraw.Draw(img_pil)
  draw.text((x, y - font_size), text, font = font)
  img = np.array(img_pil)
  return img

In [ ]:
min_confidence = 40
img_copy = rgb.copy()
for i in range(0, len(result['text'])):
  confidence = int(result['conf'][i])
  if confidence > min_confidence:
    x, y, img = bouding_box(result, img_copy, i)
    text = result['text'][i]
    #cv2.putText(img_copy, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,0,255))
    img_copy = write_text(text, x, y, img_copy, font)
cv2_imshow(img_copy)

# Searching specific information

In [ ]:
import re # regular expressions

In [ ]:
img = cv2.imread('/content/table_test.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
result = pytesseract.image_to_data(rgb, config=config_tesseract, lang='por', output_type=Output.DICT)
result

In [ ]:
# https://regexr.com/
date_pattern = '^(0[1-9]|[12][0-9]|3[01])/(0[1-9]|1[012])/(19|20)\d\d$'

In [ ]:
dates = []
min_confidence = 40
img_copy = rgb.copy()
for i in range(0, len(result['text'])):
  confidence = int(result['conf'][i])
  if confidence > min_confidence:
    text = result['text'][i]
    if re.match(date_pattern, text):
      x, y, img = bouding_box(result, img_copy, i, (0,0,255))
      #cv2.putText(img_copy, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,0,255))
      img_copy = write_text(text, x, y, img_copy, font, 12)
      dates.append(text)
    else:
      x, y, img_copy = bouding_box(result, img_copy, i)
cv2_imshow(img_copy)

In [ ]:
dates

# Detecting texts in natural scenarios

In [ ]:
img = cv2.imread('/content/cup.jpg')
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2_imshow(rgb)

In [ ]:
config_tesseract = "--tessdata-dir tessdata --psm 11"
result = pytesseract.image_to_data(rgb, config=config_tesseract, lang = 'eng', output_type=Output.DICT)
result

In [ ]:
min_confidence = 40
img_copy = rgb.copy()
for i in range(0, len(result['text'])):
  confidence = int(result['conf'][i])
  if confidence > min_confidence:
    text = result['text'][i]
    if not text.isspace() and len(text) > 0:
      x, y, img = bouding_box(result, img_copy, i)
      cv2.putText(img_copy, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,0,255))
cv2_imshow(img_copy)

In [ ]:
result['conf']

In [ ]:
result['text']